# 🎃 ZapalloAI — Notebook 02: Preprocesamiento y Unificación

**Universidad de las Fuerzas Armadas ESPE**

Este notebook:
1. Unifica las clases de los dos datasets con el mapeo definido
2. Elimina duplicados
3. Realiza un split estratificado **70% train / 15% val / 15% test**
4. Aplica augmentations offline con Albumentations
5. Genera estadísticas del dataset final
6. Guarda el dataset procesado en Drive

## Mapeo de clases
| Clase objetivo | Sweet Pumpkin | Cucurbit Leaf |
|---|---|---|
| `healthy` | Healthy | healthy |
| `downy_mildew` | Powdery Mildew | downy_mildew |
| `leaf_curl` | Leaf Curl | leaf_curl_virus |
| `mosaic_virus` | Mosaic Virus | mosaic |
| `red_beetle` | Red Beetle | *(no existe)* |

In [ ]:
# ── 0. Instalar dependencias ──────────────────────────────────────
!pip install -q ultralytics albumentations imagehash
print('✅ Dependencias instaladas')

In [ ]:
# ── 1. Configuración ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, random
import numpy as np
from pathlib import Path
from PIL import Image
import imagehash

DRIVE_BASE  = '/content/drive/MyDrive/ZapalloAI'
SWEET_DIR   = f'{DRIVE_BASE}/sweet_pumpkin'
CUCURB_DIR  = f'{DRIVE_BASE}/cucurbit_leaf'
OUTPUT_DIR  = f'{DRIVE_BASE}/dataset_processed'

CLASSES = ['healthy', 'downy_mildew', 'leaf_curl', 'mosaic_virus', 'red_beetle']

# Mapeo: nombre en disco → nombre normalizado
# ⚠️ Ajustar si los nombres de carpetas en tu Drive son diferentes
CLASS_MAP = {
    # Sweet Pumpkin Disease Recognition
    'Healthy':       'healthy',
    'Powdery Mildew':'downy_mildew',
    'Leaf Curl':     'leaf_curl',
    'Mosaic Virus':  'mosaic_virus',
    'Red Beetle':    'red_beetle',
    # Cucurbit Leaf Disease Dataset (ajustar si es necesario)
    'healthy':          'healthy',
    'Healthy':          'healthy',
    'downy_mildew':     'downy_mildew',
    'leaf_curl_virus':  'leaf_curl',
    'leaf_curl':        'leaf_curl',
    'mosaic':           'mosaic_virus',
    'mosaic_virus':     'mosaic_virus',
}

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# TEST = 1 - 0.70 - 0.15 = 0.15
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('✅ Configuración lista')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# ── 2. Eliminar duplicados entre datasets ─────────────────────────
def get_all_images(base_dir: str) -> list:
    base = Path(base_dir)
    if not base.exists():
        return []
    return list(base.rglob('*.jpg')) + list(base.rglob('*.png')) + list(base.rglob('*.jpeg'))

def compute_hashes(images: list) -> dict:
    """Calcula pHash para una lista de imágenes."""
    hashes = {}
    for i, img_path in enumerate(images):
        if i % 500 == 0 and i > 0:
            print(f'   {i}/{len(images)}...')
        try:
            with Image.open(img_path) as img:
                hashes[str(img_path)] = imagehash.phash(img)
        except Exception:
            pass
    return hashes

print('🔍 Calculando hashes de imágenes...')
sweet_imgs  = get_all_images(SWEET_DIR)
cucurb_imgs = get_all_images(CUCURB_DIR)
all_imgs    = sweet_imgs + cucurb_imgs

print(f'Total imágenes: {len(all_imgs):,}')
hashes = compute_hashes(all_imgs)

# Eliminar exactos duplicados (hash idéntico)
seen = {}
to_skip = set()
for path, h in hashes.items():
    key = str(h)
    if key in seen:
        to_skip.add(path)  # Saltar duplicado
    else:
        seen[key] = path

print(f'\n✅ Duplicados exactos encontrados: {len(to_skip)}')
print(f'   Imágenes únicas: {len(all_imgs) - len(to_skip):,}')

In [ ]:
# ── 3. Crear estructura de salida ─────────────────────────────────
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        os.makedirs(f'{OUTPUT_DIR}/{split}/{cls}', exist_ok=True)

print('✅ Estructura de carpetas creada:')
print(f'  {OUTPUT_DIR}/')
for split in ['train', 'val', 'test']:
    print(f'    {split}/')
    for cls in CLASSES:
        print(f'      {cls}/')

In [ ]:
# ── 4. Unificar y copiar con split estratificado ──────────────────
import pandas as pd

stats = {cls: {'train': 0, 'val': 0, 'test': 0, 'skipped': 0} for cls in CLASSES}

def process_dataset(src_dir: str, class_map: dict):
    base = Path(src_dir)
    if not base.exists():
        print(f'⚠️ No existe: {src_dir}')
        return
    
    for cls_dir in sorted(base.iterdir()):
        if not cls_dir.is_dir():
            continue
        
        dst_cls = class_map.get(cls_dir.name)
        if dst_cls is None:
            print(f'  ⚠️ Clase no mapeada: {cls_dir.name} — ignorando')
            continue
        
        # Obtener imágenes únicas (sin duplicados)
        images = [
            img for img in
            (list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')) + list(cls_dir.glob('*.jpeg')))
            if str(img) not in to_skip
        ]
        random.shuffle(images)
        
        n = len(images)
        n_train = int(n * TRAIN_RATIO)
        n_val   = int(n * VAL_RATIO)
        
        splits_map = {
            'train': images[:n_train],
            'val':   images[n_train:n_train+n_val],
            'test':  images[n_train+n_val:],
        }
        
        for split_name, split_imgs in splits_map.items():
            for img_path in split_imgs:
                # Nombre destino: evitar colisiones entre datasets
                src_stem = Path(src_dir).name[:3]  # 'swe' o 'cuc'
                dst_name = f'{src_stem}_{img_path.name}'
                dst_path = Path(f'{OUTPUT_DIR}/{split_name}/{dst_cls}/{dst_name}')
                if not dst_path.exists():
                    shutil.copy2(img_path, dst_path)
                    stats[dst_cls][split_name] += 1
        
        total_cls = sum(len(v) for v in splits_map.values())
        print(f'  ✅ {cls_dir.name} → {dst_cls}: {total_cls} imgs ')

print('📂 Procesando Sweet Pumpkin...')
process_dataset(SWEET_DIR, CLASS_MAP)

print('\n📂 Procesando Cucurbit Leaf...')
process_dataset(CUCURB_DIR, CLASS_MAP)

In [ ]:
# ── 5. Estadísticas del dataset final ────────────────────────────
import matplotlib.pyplot as plt

# Contar imágenes reales en disco
final_counts = {split: {} for split in ['train', 'val', 'test']}
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        p = Path(f'{OUTPUT_DIR}/{split}/{cls}')
        n = len(list(p.glob('*.jpg'))) + len(list(p.glob('*.png'))) + len(list(p.glob('*.jpeg')))
        final_counts[split][cls] = n

# Tabla resumen
print(f"{'Clase':<20} {'Train':>8} {'Val':>8} {'Test':>8} {'TOTAL':>8}")
print('-' * 52)
grand_total = 0
for cls in CLASSES:
    t = final_counts['train'].get(cls, 0)
    v = final_counts['val'].get(cls, 0)
    ts = final_counts['test'].get(cls, 0)
    total = t + v + ts
    grand_total += total
    print(f"{cls:<20} {t:>8} {v:>8} {ts:>8} {total:>8}")
print('-' * 52)
print(f"{'TOTAL':<20} {sum(final_counts['train'].values()):>8} "
      f"{sum(final_counts['val'].values()):>8} "
      f"{sum(final_counts['test'].values()):>8} "
      f"{grand_total:>8}")

# Verificar balance
print('\n📊 Verificación de balance de clases:')
train_counts = [final_counts['train'].get(c, 0) for c in CLASSES]
max_c = max(train_counts)
min_c = min(train_counts)
ratio = max_c / min_c if min_c > 0 else float('inf')
print(f'   Ratio max/min en train: {ratio:.2f}x')
if ratio > 3:
    print('   ⚠️ Dataset desbalanceado (>3x). Considerar oversampling.')
else:
    print('   ✅ Dataset razonablemente balanceado (<3x)')

In [ ]:
# ── 6. Augmentation offline (solo para clases minoritarias) ───────
import albumentations as A
import cv2
import numpy as np

# Pipeline de augmentation para hojas de zapallo
augment_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=45, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=40, val_shift_limit=20, p=0.5),
    A.GaussianBlur(blur_limit=(3, 7), p=0.2),
    A.GaussNoise(p=0.2),
    A.RandomShadow(p=0.3),
    A.CoarseDropout(max_holes=8, max_height=30, max_width=30, p=0.3),
])

TARGET_PER_CLASS = 1500  # Mínimo de imágenes en train por clase

aug_count = 0
for cls in CLASSES:
    train_dir = Path(f'{OUTPUT_DIR}/train/{cls}')
    existing = list(train_dir.glob('*.jpg')) + list(train_dir.glob('*.png'))
    current_count = len(existing)
    needed = max(0, TARGET_PER_CLASS - current_count)
    
    if needed == 0:
        print(f'  ✅ {cls}: {current_count} imgs (suficiente)')
        continue
    
    print(f'  🔧 {cls}: {current_count} imgs → augmentando +{needed}')
    
    generated = 0
    while generated < needed:
        src_img = random.choice(existing)
        img_bgr = cv2.imread(str(src_img))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        
        augmented = augment_pipeline(image=img_rgb)['image']
        
        out_name = f'aug_{cls}_{generated:05d}.jpg'
        out_path = train_dir / out_name
        aug_bgr = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
        cv2.imwrite(str(out_path), aug_bgr, [cv2.IMWRITE_JPEG_QUALITY, 90])
        
        generated += 1
    
    aug_count += generated

print(f'\n✅ Augmentation completado: +{aug_count} imágenes generadas')

In [ ]:
# ── 7. Guardar dataset.yaml actualizado ───────────────────────────
yaml_content = f"""# Dataset ZapalloAI — generado por Notebook 02
# Fecha: $(date)
path: {OUTPUT_DIR}
train: train
val: val
test: test

nc: 5
names:
  0: healthy
  1: downy_mildew
  2: leaf_curl
  3: mosaic_virus
  4: red_beetle
"""

yaml_path = f'{DRIVE_BASE}/dataset_final.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f'✅ dataset_final.yaml guardado en: {yaml_path}')
print(f'\n🚀 SIGUIENTE: Ejecutar Notebook 03 con:')
print(f'   data={OUTPUT_DIR}')

# Estadísticas finales
total_final = sum(
    len(list(Path(f'{OUTPUT_DIR}/{split}/{cls}').glob('*.*')))
    for split in ['train', 'val', 'test']
    for cls in CLASSES
)
print(f'\n📊 Total dataset final: {total_final:,} imágenes')